In [1]:
import pandas as pd
import numpy as np

# Define the file path from your system
file_path = r"C:\Users\pc\Documents\Downloads\DataCo Supply Chain\DataCoSupplyChainDataset.csv"

# Load the dataset with the appropriate encoding
df = pd.read_csv(file_path, encoding='latin1')

# 1. Print the shape of the dataset (Rows and Columns)
print("Dataset Shape (Rows, Columns):", df.shape)
print("-" * 50)

# 2. Display all column names in the dataset
print("Column Names in the Dataset:")
print(df.columns.tolist())
print("-" * 50)

# 3. Check for missing values (Nulls) and sort them in descending order
print("Columns with missing values and their counts:")
missing_data = df.isnull().sum()
print(missing_data[missing_data > 0].sort_values(ascending=False))

Dataset Shape (Rows, Columns): (180519, 53)
--------------------------------------------------
Column Names in the Dataset:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Categ

In [2]:
# 1. Drop completely empty or unnecessary columns
columns_to_drop = ['Product Description', 'Order Zipcode', 'Customer Password', 'Customer Email']
df_cleaned = df.drop(columns=columns_to_drop)

# 2. Handle remaining missing values
df_cleaned['Customer Lname'] = df_cleaned['Customer Lname'].fillna('Unknown')
df_cleaned['Customer Zipcode'] = df_cleaned['Customer Zipcode'].fillna(0) # or replace with placeholder

# 3. Verify that there are no more missing values in the critical columns
print("Remaining missing values:")
print(df_cleaned.isnull().sum().sum())

# 4. Check the new shape of the dataset
print("New dataset shape:", df_cleaned.shape)

Remaining missing values:
0
New dataset shape: (180519, 49)


In [3]:
# 1. Create Customers Dimension Table
customer_cols = ['Customer Id', 'Customer Fname', 'Customer Lname', 'Customer Segment', 
                 'Customer City', 'Customer State', 'Customer Country', 'Latitude', 'Longitude']
dim_customers = df_cleaned[customer_cols].drop_duplicates(subset=['Customer Id'])

# 2. Create Products Dimension Table
product_cols = ['Product Card Id', 'Product Name', 'Product Category Id', 'Category Name', 'Product Price', 'Department Id', 'Department Name']
dim_products = df_cleaned[product_cols].drop_duplicates(subset=['Product Card Id'])

# 3. Create Shipping/Logistics Dimension Table
# We will create a unique Shipping ID using the index to create a dimension for shipping modes
dim_shipping = df_cleaned[['Shipping Mode', 'Delivery Status', 'Late_delivery_risk']].drop_duplicates().reset_index(drop=True)
dim_shipping['Shipping_Key'] = dim_shipping.index + 1

# Merge the Shipping_Key back to the main dataframe to use it in the Fact table
df_cleaned = df_cleaned.merge(dim_shipping, on=['Shipping Mode', 'Delivery Status', 'Late_delivery_risk'], how='left')

# 4. Create the Fact Table (Orders and Sales)
# This table will only keep IDs to connect with Dimensions, plus the numerical values (Sales, Profit, Dates)
fact_cols = ['Order Id', 'Order Customer Id', 'Product Card Id', 'Shipping_Key',
             'order date (DateOrders)', 'shipping date (DateOrders)', 'Days for shipping (real)', 'Days for shipment (scheduled)',
             'Sales', 'Order Item Quantity', 'Benefit per order', 'Order Item Discount', 'Order Item Profit Ratio']
fact_sales = df_cleaned[fact_cols]

# 5. Verify the shapes of our new tables
print("Dim Customers Shape:", dim_customers.shape)
print("Dim Products Shape:", dim_products.shape)
print("Dim Shipping Shape:", dim_shipping.shape)
print("Fact Sales Shape:", fact_sales.shape)

Dim Customers Shape: (20652, 9)
Dim Products Shape: (118, 7)
Dim Shipping Shape: (12, 4)
Fact Sales Shape: (180519, 13)


In [4]:
import os

# Define the output directory (same as your data folder)
output_path = r"C:\Users\pc\Documents\Downloads\DataCo Supply Chain\\"

# Save the dataframes to CSV files
dim_customers.to_csv(output_path + 'Dim_Customers.csv', index=False, encoding='utf-8')
dim_products.to_csv(output_path + 'Dim_Products.csv', index=False, encoding='utf-8')
dim_shipping.to_csv(output_path + 'Dim_Shipping.csv', index=False, encoding='utf-8')
fact_sales.to_csv(output_path + 'Fact_Sales.csv', index=False, encoding='utf-8')

print("All tables have been exported successfully as CSV files!")
print("Files location:", output_path)

All tables have been exported successfully as CSV files!
Files location: C:\Users\pc\Documents\Downloads\DataCo Supply Chain\\
